# 📦 pip install - Package Installation

**"Standing on the shoulders of giants."**

`pip` (Package Installer for Python) is the standard tool for installing third-party libraries from PyPI (Python Package Index). Understanding `pip install` is fundamental to modern Python development.

## 🎯 Learning Objectives

- Understand how `pip install` resolves dependencies
- Master version specifiers and constraints
- Learn the difference between local and global installations
- Explore the internal mechanisms of package resolution
- Identify common pitfalls and how to avoid them

## 1. The Theory: How pip Works

### Package Ecosystem

When you run `pip install requests`, here's what happens:

1. **Query PyPI**: Connects to `https://pypi.org/pypi/<package>/json`
2. **Resolve Dependencies**: Reads `setup.py` or `pyproject.toml` metadata
3. **Dependency Resolution**: Uses a SAT solver to find compatible versions
4. **Download Wheels**: Prefers binary `.whl` over source `.tar.gz`
5. **Install to site-packages**: Unpacks to `lib/pythonX.Y/site-packages/`

### Why This Matters

Without `pip`, you'd need to:
- Manually download every library
- Track down all transitive dependencies
- Build C extensions yourself
- Manage version conflicts manually

Real-world impact: A single `pip install tensorflow` triggers installation of ~40+ dependencies automatically.

## 2. Basic Usage (Beginner Level)

The simplest form installs the latest version.

In [ ]:
# Install a package (run in terminal, shown here for reference)
# pip install requests

# Verify installation
import requests
print(f"requests version: {requests.__version__}")
print(f"Installed at: {requests.__file__}")

In [ ]:
# View package metadata
import pkg_resources

pkg = pkg_resources.get_distribution("requests")
print(f"Name: {pkg.project_name}")
print(f"Version: {pkg.version}")
print(f"Location: {pkg.location}")

# List dependencies
print("\nDependencies:")
for req in pkg.requires():
    print(f"  - {req}")

## 3. Intermediate: Version Constraints

Control exactly which version gets installed using PEP 440 specifiers.

In [ ]:
# Version specifier syntax (for terminal use)
"""
pip install requests==2.28.0      # Exact version (pinned)
pip install requests>=2.20.0      # Minimum version
pip install requests~=2.28.0      # Compatible release (2.28.x)
pip install 'requests>=2.20,<3.0' # Range (note quotes for shell)
"""

# Check what version you have
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "requests"],
    capture_output=True,
    text=True
)

# Parse output
for line in result.stdout.split('\n'):
    if line.startswith('Version:'):
        print(line)

## 4. Advanced: Understanding the Resolution Algorithm

### The Dependency Hell Problem

Consider this scenario:
- Package A requires `numpy>=1.20`
- Package B requires `numpy<1.22`
- pip must find a version that satisfies BOTH

Solution: `numpy==1.21.x` (and pip will automatically detect this)

### Inspecting What Gets Installed

In [ ]:
# Simulate what pip would install (dry-run)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--dry-run", "requests"],
    capture_output=True,
    text=True
)

print("Would install:")
print(result.stdout)

### Pro Tip: The Resolver in Action

Since pip 20.3 (2020), pip uses a **backtracking resolver**. This means:
- It tries different combinations until it finds a valid solution
- Can be SLOW for complex dependency trees (minutes for large projects)
- Use `--use-deprecated=legacy-resolver` to revert to old behavior (not recommended)

In [ ]:
# Check pip version (resolver was improved in 20.3+)
import pip
print(f"pip version: {pip.__version__}")

# Recommended: Always upgrade pip itself
# python -m pip install --upgrade pip

## 5. Common Pitfalls & Debugging

### Pitfall 1: System vs. Virtual Environment

**Problem**: Installing to system Python pollutes global namespace.

```bash
# BAD: Installs globally
sudo pip install <package>  # Never use sudo with pip!

# GOOD: Use venv
python -m venv myenv
source myenv/bin/activate  # Windows: myenv\Scripts\activate
pip install <package>
```

### Pitfall 2: Conflicting Versions

**Error Message**: `ERROR: Cannot install X and Y because these package versions have conflicting dependencies.`

**Solution**: Use `pip install --use-pep517` for better error messages, or check `pip install --dry-run` first.

In [ ]:
# Debugging: Check for conflicts in current environment
result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ No conflicts detected")
else:
    print("❌ Conflicts found:")
    print(result.stdout)

### Pitfall 3: Cached Packages

**Problem**: pip caches downloads. If PyPI had a corrupted file, your cache might have it.

**Solution**: `pip install --no-cache-dir <package>` or `pip cache purge`

## 6. Best Practices (Industry Standard)

### 1. Always Use Virtual Environments
```bash
python -m venv .venv
```

### 2. Pin Versions in Production
```txt
# requirements.txt
requests==2.28.0  # Exact pin
numpy>=1.20,<2.0  # Major version lock
```

### 3. Use `requirements.txt` for Reproducibility
```bash
pip freeze > requirements.txt  # Export current env
pip install -r requirements.txt  # Reproduce elsewhere
```

### 4. Consider Alternative Tools
- **Poetry**: Better dependency resolution + lock files
- **Pipenv**: Combines pip + venv
- **uv**: Rust-based, 10-100x faster than pip

### 5. Security: Verify Packages
```bash
pip install --require-hashes -r requirements.txt
```

## 7. Real-World Scenario

**Scenario**: You're deploying a Flask web app to production.

**Anti-Pattern**:
```bash
pip install flask  # Installs latest, breaks in 6 months
```

**Professional Pattern**:
```bash
# Development
pip install flask
pip freeze > requirements.txt

# Production (deterministic install)
pip install -r requirements.txt --no-deps --require-hashes
```

This ensures:
- Exact same versions in dev/staging/prod
- No surprise upgrades break your app
- Security via hash verification

## 8. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **pip install** | Downloads and installs packages from PyPI |
| **Dependency Resolution** | pip uses backtracking to find compatible versions |
| **Version Specifiers** | Use `==`, `>=`, `~=` to control versions |
| **Virtual Environments** | ALWAYS use venv to isolate projects |
| **requirements.txt** | Lock versions for reproducible builds |

### Next Steps
- Learn about `pip list` and `pip show`
- Explore `pip install --editable` for local development
- Study `pyproject.toml` (PEP 517/518) as the modern alternative to `setup.py`